In [ ]:
from pathlib import Path
import json
from typing import List

import numpy as np
import pandas as pd
import h5py

from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
repo_dir = Path("../../..")
results_dir = repo_dir / 'extract_n_eval' / "results" / "proj_feats"
save_dir = repo_dir / "visualization" / "paper" / "supp" / "figures"

assert results_dir.exists(), f"data directory {results_dir} does not exist!"


In [ ]:
valid_filenames = [
    "bs_fz.json",
    "bs_mh.json",
    "tvsd.json",
    "things_fmri.json",
    "things_eeg1.json",
    "things_eeg2.json",
    "things_meg.json",
    "nsd_func1pt8mm_individualROIs.json",
    "nsd_fsaverage_individualROIs.json",
    "nsd_nativesurface_individualROIs.json",
]

found_results_files = [file for file in results_dir.glob("**/*.json") if file.name in valid_filenames]

all_results = []

for file in tqdm(found_results_files):
    try:
        with open(file, 'r') as f:
            result = json.load(f)
    except Exception as e:
        print(f"Error loading {file}: {e}")
        continue
    all_results.extend(result["results"])
    
df_all_results = pd.DataFrame(all_results)
df_all_results

In [ ]:

# df_all_results = df_all_results[df_all_results.roi != "occipital_parietal"]
df_all_results = df_all_results[df_all_results.roi != "IT_glasser"]

df_all_results.loc[df_all_results.roi=='occipital_parietal', 'roi'] = 'Occipital+Parietal'
df_all_results.loc[df_all_results.roi=='whole_brain', 'roi'] = 'Whole Brain'
# df_all_results.roi = df_all_results.roi.apply(lambda x: x.capitalize())

In [ ]:
best_layers = []
metrics = [
    # "cv_score",
    # "rsa_c_train",
    # "cka_c_train",
    
    "pearsonr_nc",
    "rsa_c_test",
    "cka_c_test",
    
    # "rsa_ve_train",
    "rsa_ve_test",
    # "cka_ve_train",
    "cka_ve_test",
    
    # 'pearsonr_wrong'
]
sorting_metric = "pearsonr_nc"

df_all_results['pearsonr_wrong'] = df_all_results['pearsonr']**2 / df_all_results['noise_ceiling']**4

df_avg_subjects = df_all_results.groupby(['model_id', 'layer_name', 'benchmark_name', 'roi']).agg({metric: 'mean' for metric in metrics}).reset_index()
df_avg_subjects


best_layers = []
for (model_id, benchmark_name, roi), df_group in tqdm(df_avg_subjects.groupby(['model_id', 'benchmark_name', 'roi'])):
    df_avg_subjects_sorted = df_group.sort_values(by=sorting_metric, ascending=False)
    best_layer = df_avg_subjects_sorted.iloc[0]
    best_layers.append(best_layer)
    
df_best_layers = pd.DataFrame(best_layers)
df_best_layers



df_becnhmark_averages = df_best_layers.groupby(['model_id', 'layer_name', 'benchmark_name']).agg({metric: 'mean' for metric in metrics}).reset_index()
df_becnhmark_roi_averages = df_best_layers.groupby(['model_id', 'layer_name', 'benchmark_name', 'roi']).agg({metric: 'mean' for metric in metrics}).reset_index()
df_becnhmark_averages
    


In [ ]:
df_best_layers.roi.unique()

In [ ]:
#  print top 10 models in each benchmark
# for benchmark_name, df_group in df_becnhmark_averages.groupby('benchmark_name'):
#     print(f"Benchmark: {benchmark_name}")
#     df_group_sorted = df_group.sort_values(by=sorting_metric, ascending=False)
#     print(df_group_sorted[['model_id', 'layer_name', sorting_metric]].head(10))
#     print("\n")
    
# # print top 10 models in each benchmark and roi
# for (benchmark_name, roi), df_group in df_becnhmark_roi_averages.groupby(['benchmark_name', 'roi']):
#     print(f"Benchmark: {benchmark_name}, ROI: {roi}")
#     df_group_sorted = df_group.sort_values(by=sorting_metric, ascending=False)
#     print(df_group_sorted[['model_id', 'layer_name', sorting_metric]].head(10))
#     print("\n")

In [ ]:
MODEL_NAME = "deit_small_imagenet_full_seed-0"
# MODEL_NAME = "vit_base_patch16_dinov3.lvd1689m"
# MODEL_NAME = "resnet50_imagenet_full"
# MODEL_NAME = "deitflex-l-1_imagenet_full_seed-0"
# MODEL_NAME = "deit_small_imagenet_100_seed-0"

X_KEY = "layer_position_normalized"
Y_KEYS = [
    "pearsonr_nc",
    # "cv_score",
    # "rsa_c_train",
    "rsa_c_test",
    # "cka_c_train",
    "cka_c_test",
    
    # "rsa_ve_train",
    "rsa_ve_test",
    # "cka_ve_train",
    "cka_ve_test",
]

FN_FILTER_LAYERS = lambda x: (
    False
    # or x.endswith("norm1") 
    or x.endswith("norm2") 
    # or x.endswith("mlp") 
    # or x.endswith("attn")
)
# FN_FILTER_LAYERS = lambda x: True  # No filter

In [ ]:
METRIC_NAMES_MAP = {
    "pearsonr_nc": "Pearson's r (Noise ceiled)",
    "cv_score": "Cross-validated score (NC)",
    "cka_c_test": "CKA",
    "rsa_c_test": "RSA",
    "cka_ve_test": "CKA-VE",
    "rsa_ve_test": "RSA-VE",
    # "cka_c_test": "CKA-C (test)",
    # "rsa_c_test": "RSA-C (test)",
    # "cka_ve_test": "CKA-VE (test)",
    # "rsa_ve_test": "RSA-VE (test)",
    "cka_c_train": "CKA-C (train)",
    "rsa_c_train": "RSA-C (train)",
    "cka_ve_train": "CKA-VE (train)",
    "rsa_ve_train": "RSA-VE (train)",
}

BENCHMARK_NAMES_MAP = {
    "nsd_fsaverage_individualROIs": "NSD fMRI - fsaverage",
    "nsd_func1pt8mm_individualROIs": "NSD fMRI - Native Volume",
    "nsd_nativesurface_individualROIs": "NSD fMRI - Native Surface",
    "things_fmri": "THINGS fMRI",
    "things_eeg1": "THINGS EEG-1",
    "things_eeg2": "THINGS EEG-2",
    "things_meg": "THINGS MEG",
    "tvsd": "TVSD",
    "bs_fz": "Brain-Score FZ",
    "bs_mh": "Brain-Score MH",
}

BENCHMARK_NAMES_MAP = {
    "nsd_fsaverage_individualROIs": "NSD fMRI - fsaverage",
    "nsd_func1pt8mm_individualROIs": "NSD-fMRI",
    "nsd_nativesurface_individualROIs": "NSD fMRI - Native Surface",
    "things_fmri": "T-fMRI",
    "things_eeg1": "T-EEG1",
    "things_eeg2": "T-EEG2",
    "things_meg": "T-MEG",
    "tvsd": "TVSD-EP",
    "bs_fz": "FZ-EP",
    "bs_mh": "MH-EP",
}

In [ ]:
def get_metric_colors(metrics: List[str]) -> dict:
    base_colors = sns.color_palette("tab10", n_colors=len(metrics))
    metric_colors = {metric: base_colors[i] for i, metric in enumerate(metrics)}
    return metric_colors

In [ ]:

def plot_layerwise_scores(
    df: pd.DataFrame,
    rois: List[str] = None,
    y_keys: List[str] = None,
    x_key: str = "layer_position_normalized",
    benchmark_name: str = None,
    nrows: int = 2,
    ncols: int = 3,
    subfigsize: tuple = (5, 5),
    zoom: int = 1,
    dpi=300,
    ):
    
    
    metric_colors = get_metric_colors(y_keys)

    # Keep only ROIs that exist in df, preserve order
    rois = [r for r in rois if r in df["roi"].unique()]

    # Compute subplot grid
    assert len(rois) <= nrows * ncols, f"Not enough subplots for the number of ROIs, got {len(rois)} ROIs but only {nrows * ncols} subplots."
    
    df['layer_position_normalized'] = df['layer_position'] / df['layer_position'].max()

    figure_size = (subfigsize[0] * ncols * zoom, subfigsize[1] * nrows * zoom)
    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=figure_size,
        sharex=True,
        sharey=True,
        dpi=dpi
    )

    for idx, ax, in enumerate(axes.flatten()):
        
        if idx >= len(rois):
            ax.remove()
            continue
        
        roi = rois[idx]

        df_roi = df[df["roi"] == roi]

        for y_key in y_keys:
            columns = list(set(["layer_position", "layer_name", x_key]))
            agg = df_roi.groupby(columns).agg(
                mean=(y_key, "mean"),
                stderr=(y_key, "sem")
            ).reset_index()
            agg = agg.sort_values(by=x_key)

            # Line plot
            ax.plot(
                agg[x_key],
                agg["mean"],
                label=f"{METRIC_NAMES_MAP.get(y_key, y_key)}",
                color=metric_colors[y_key],
                linewidth=2,
            )

            # SE shading
            ax.fill_between(
                agg[x_key],
                agg["mean"] - agg["stderr"],
                agg["mean"] + agg["stderr"],
                color=metric_colors[y_key],
                alpha=0.15
            )

        ax.set_title(f"ROI: {roi}", fontsize=16, fontweight="bold")
        ax.set_xlabel("Layer position", fontsize=12, fontweight="bold")
        # ax.set_ylabel("NC Correlation (mean ± SE)", fontsize=12, fontweight="bold")
        ax.set_ylabel("Score (mean ± SE)", fontsize=12, fontweight="bold")
        ax.spines[['top', 'right']].set_visible(False)
        ax.grid(visible=True, linestyle='--', alpha=0.5)
        
        ax.set_ylim(0, 1.0)
        
        
        ax.legend()

    # Hide unused axes (if ROIs < total slots)
    for i in range(len(rois), len(axes.flatten())):
        axes.flatten()[i].axis("off")

    if benchmark_name:
        benchmark_title = BENCHMARK_NAMES_MAP.get(benchmark_name, benchmark_name)
        plt.suptitle(f"{benchmark_title}", fontsize=20, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.97])

    return fig, axes

In [ ]:
def save_figs(fig, save_dir, base_filename, dpi=300, formats=("png", "pdf", "svg")):
    for fmt in formats:
    
        save_fmt_dir = Path(save_dir) / fmt
        if not save_fmt_dir.exists():
            save_fmt_dir.mkdir(parents=True, exist_ok=False)
        
        file_path = save_fmt_dir / f"{base_filename}.{fmt}"
        
        fig.savefig(file_path, dpi=dpi, bbox_inches='tight')

        print(f"Figure saved to {file_path}")

# Brain-Score FreemanZiemba

In [ ]:
df_all_results.columns

In [ ]:
model_id = MODEL_NAME
benchmark_name = "bs_fz"
x_key = X_KEY
y_keys = Y_KEYS
fn_filter_layers = FN_FILTER_LAYERS

df = df_all_results[df_all_results['model_id'] == model_id].copy()
df = df[df["benchmark_name"] == benchmark_name]
data = df[df['layer_name'].apply(fn_filter_layers)].copy()
rois = data["roi"].unique().tolist()

fig, axes = plot_layerwise_scores(df=data, rois=rois, y_keys=y_keys, x_key=x_key, benchmark_name=benchmark_name, nrows=1, ncols=2, subfigsize=(6,4), zoom=1, dpi=300)

save_figs(
    fig,
    save_dir=save_dir,
    base_filename="layer_hierarchy-bs_fz",
    dpi=300,
    formats=("png", "pdf", "svg"),
)


# Brain-Score MajajHong

In [ ]:
model_id = MODEL_NAME
benchmark_name = "bs_mh"
x_key = X_KEY
y_keys = Y_KEYS
fn_filter_layers = FN_FILTER_LAYERS

df = df_all_results[df_all_results['model_id'] == model_id].copy()
df = df[df["benchmark_name"] == benchmark_name]
data = df[df['layer_name'].apply(fn_filter_layers)].copy()
rois = data["roi"].unique().tolist()

fig, axes = plot_layerwise_scores(df=data, rois=rois, y_keys=y_keys, x_key=x_key, benchmark_name=benchmark_name, nrows=1, ncols=2, subfigsize=(6,4), zoom=1, dpi=300)

save_figs(
    fig,
    save_dir=save_dir,
    base_filename="layer_hierarchy-bs_mh",
    dpi=300,
    formats=("png", "pdf", "svg"),
)


# TVSD

In [ ]:
model_id = MODEL_NAME
benchmark_name = "tvsd"
x_key = X_KEY
y_keys = Y_KEYS
fn_filter_layers = FN_FILTER_LAYERS

df = df_all_results[df_all_results['model_id'] == model_id].copy()
df = df[df["benchmark_name"] == benchmark_name]
if len(df) == 0:
    print("No data available yet.")
else:
    data = df[df['layer_name'].apply(fn_filter_layers)].copy()
    rois = data["roi"].unique().tolist()
    fig, axes = plot_layerwise_scores(df=data, rois=rois, y_keys=y_keys, x_key=x_key, benchmark_name=benchmark_name, nrows=1, ncols=3, subfigsize=(6,6), zoom=0.75, dpi=300)
    
save_figs(
    fig,
    save_dir=save_dir,
    base_filename="layer_hierarchy-tvsd",
    dpi=300,
    formats=("png", "pdf", "svg"),
)

# THINGS-EEG1

In [ ]:
model_id = MODEL_NAME
benchmark_name = "things_eeg1"
x_key = X_KEY
y_keys = Y_KEYS
fn_filter_layers = FN_FILTER_LAYERS

df = df_all_results[df_all_results['model_id'] == model_id].copy()
df = df[df["benchmark_name"] == benchmark_name]
if len(df) == 0:
    print("No data available yet.")
else:
    data = df[df['layer_name'].apply(fn_filter_layers)].copy()
    rois = data["roi"].unique().tolist()

    fig, axes = plot_layerwise_scores(df=data, rois=rois, y_keys=y_keys, x_key=x_key, benchmark_name=benchmark_name, nrows=2, ncols=4, subfigsize=(6,4), zoom=0.75, dpi=300)

save_figs(
    fig,
    save_dir=save_dir,
    base_filename="layer_hierarchy-eeg1",
    dpi=300,
    formats=("png", "pdf", "svg"),
)

# THINGS-EEG2

In [ ]:
model_id = MODEL_NAME
benchmark_name = "things_eeg2"
x_key = X_KEY
y_keys = Y_KEYS
fn_filter_layers = FN_FILTER_LAYERS

df = df_all_results[df_all_results['model_id'] == model_id].copy()
df = df[df["benchmark_name"] == benchmark_name]
if len(df) == 0:
    print("No data available yet.")
else:
    data = df[df['layer_name'].apply(fn_filter_layers)].copy()
    rois = data["roi"].unique().tolist()

    fig, axes = plot_layerwise_scores(df=data, rois=rois, y_keys=y_keys, x_key=x_key, benchmark_name=benchmark_name, nrows=2, ncols=4, subfigsize=(6,4), zoom=0.75, dpi=300)
    
save_figs(
    fig,
    save_dir=save_dir,
    base_filename="layer_hierarchy-eeg2",
    dpi=300,
    formats=("png", "pdf", "svg"),
)

# THINGS-MEG

In [ ]:
model_id = MODEL_NAME
benchmark_name = "things_meg"
x_key = X_KEY
y_keys = Y_KEYS
fn_filter_layers = FN_FILTER_LAYERS

df = df_all_results[df_all_results['model_id'] == model_id].copy()
df = df[df["benchmark_name"] == benchmark_name]
if len(df) == 0:
    print("No data available yet.")
else:
    data = df[df['layer_name'].apply(fn_filter_layers)].copy()
    rois = data["roi"].unique().tolist()

    fig, axes = plot_layerwise_scores(df=data, rois=rois, y_keys=y_keys, x_key=x_key, benchmark_name=benchmark_name, nrows=2, ncols=4, subfigsize=(6,4), zoom=0.75, dpi=300)
    
save_figs(
    fig,
    save_dir=save_dir,
    base_filename="layer_hierarchy-meg",
    dpi=300,
    formats=("png", "pdf", "svg"),
)

# THINGS-fMRI

In [ ]:
model_id = MODEL_NAME
benchmark_name = "things_fmri"
x_key = X_KEY
y_keys = Y_KEYS
fn_filter_layers = FN_FILTER_LAYERS

df = df_all_results[df_all_results['model_id'] == model_id].copy()
df = df[df["benchmark_name"] == benchmark_name]
if len(df) == 0:
    print("No data available yet.")
else:
    data = df[df['layer_name'].apply(fn_filter_layers)].copy()
    rois = data["roi"].unique().tolist()
    fig, axes = plot_layerwise_scores(df=data, rois=rois, y_keys=y_keys, x_key=x_key, benchmark_name=benchmark_name, nrows=7, ncols=4, subfigsize=(6,4), zoom=0.75, dpi=300)
    
    
save_figs(
    fig,
    save_dir=save_dir,
    base_filename="layer_hierarchy-tfmri",
    dpi=300,
    formats=("png", "pdf", "svg"),
)

# NSD

In [ ]:
model_id = MODEL_NAME
benchmark_name = "nsd_func1pt8mm_individualROIs"
x_key = X_KEY
y_keys = Y_KEYS
fn_filter_layers = FN_FILTER_LAYERS

df = df_all_results[df_all_results['model_id'] == model_id].copy()
df = df[df["benchmark_name"] == benchmark_name]
if len(df) == 0:
    print("No data available yet.")
else:
    data = df[df['layer_name'].apply(fn_filter_layers)].copy()
    rois = data["roi"].unique().tolist()
    fig, axes = plot_layerwise_scores(df=data, rois=rois, y_keys=y_keys, x_key=x_key, benchmark_name=benchmark_name, nrows=7, ncols=4, subfigsize=(6,4), zoom=0.75, dpi=300)
    
    
save_figs(
    fig,
    save_dir=save_dir,
    base_filename="layer_hierarchy-nsd",
    dpi=300,
    formats=("png", "pdf", "svg"),
)